<a href="https://colab.research.google.com/github/Pawan-model/Huggingface-Experiment/blob/main/Chapter_05_Datasets/Semantic_search_with_FAISS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers
!pip install transformers[sentencepiece]

In [ ]:
from datasets import load_dataset

issues_dataset = load_dataset("lewtun/github-issues", split="train")
issues_dataset

In [ ]:
issues_dataset=issues_dataset.filter(lambda x: (x['is_pull_request']==False and len(x['comments'])>0))
issues_dataset

In [ ]:
columns=issues_dataset.column_names
columns_to_keep=['title','body','html_url','comments']
columns_to_remove=set(columns_to_keep).symmetric_difference(columns)
issues_dataset=issues_dataset.remove_columns(columns_to_remove)
issues_dataset

In [ ]:
#our comments column is currently a list of comments for each issue, we need to “explode” the column
#so that each row consists of an (html_url, title, body, comment) tuple. In Pandas we can do this with the
#DataFrame.explode() function, which creates a new row for each element in a list-like column, while replicating all the other column values.
issues_dataset.set_format('pandas')
df=issues_dataset[:]
comments_df=df.explode('comments',ignore_index=True)


In [ ]:
from datasets import Dataset
comments_dataset=Dataset.from_pandas(comments_df)
comments_dataset

In [ ]:
comments_dataset=comments_dataset.map(lambda x: {'comment_length': len(x['comments'].split())})
comments_dataset=comments_dataset.filter(lambda x: x['comment_length']>15)
comments_dataset

In [ ]:
def concatenate_text(examples):
  return {'text':
          examples['title']
          +'/n'
          +examples['comments']
          +'/n'
          +examples['body']}
comments_dataset=comments_dataset.map(concatenate_text)